# EP-KD run: vgg13

Original EP-KD top3 CAM curriculum wrapped around the **official method objectives**.
DKD keeps CE + decoupled KD; ReviewKD keeps CE + ABF/HCL; SimKD keeps feature-only
alignment and the reused teacher head (no added CE or vanilla-KD loss).
Full-image and crop objectives are summed. No TGSC quality gate or pixel cache is used.

Default full recipe:16/20/24/28/32 at48epochs each, stride8, passes3/3/3/1/1,
full-image anchor, full-image pad4/flip and crop pad2/flip. Lower-ranked28px cached
members repeat because only one grid window exists; the separate difficulty experiment
is deliberately not mixed into this integration. Trainable crop-path BatchNorm running
statistics are held fixed. FP32 implementation; original alignment and warmup preserved.

Setup clones the published code from GitHub at a pinned commit; no runtime Dataset or ZIP is needed.
On Kaggle enable Internet and a GPU.
Original CIFAR training data is required; you can attach cifar-100-python/ or download it.
Existing teacher-only inputs work, but original TGSC crop banks are not consumed.
The verification notebook uses explicit short settings; full notebooks retain240epochs.
Logs/CSV/PNG/SVG, latest/best checkpoints and teacher-free exports are saved in OUTPUT.
Evaluation is every5epochs plus final/requested stop. Resume rejects changed science/data/code.


In [ ]:
from pathlib import Path
REPO = Path("/kaggle/working/mdistiller_epkd_github")
REPO_URL = "https://github.com/ahmad903501/tgsc-mdistiller.git"
REVISION = '49113223d7c98422888321fbad8ff410953e4a46'
REQUIRED_SOURCE_FILES = ["tools/run_epkd.py", "mdistiller/epkd/runtime.py"]
DATA_ROOT = Path("/kaggle/working/data")
OUTPUT = Path("/kaggle/working/epkd_runs")
TEACHER = 'vgg13'
TEACHER_CHECKPOINT = None  # Optional attached checkpoint; otherwise discover or download it.
CONFIGS = ['simkd_vgg13_to_vgg8.json']
GPU = 0  # Each process uses one GPU. Run another independent notebook on the other GPU if desired.
SEEDS = [27]
EVAL_EVERY = 5
TEST_DATA = None  # Optional verified cifar100-test.npz.


In [ ]:
from pathlib import Path
import json, subprocess, sys

# Clone the published implementation; no runtime ZIP/Dataset is needed.
def complete(root):
    return all((root/path).is_file() for path in REQUIRED_SOURCE_FILES)

cloned_now = False
if not (REPO/".git").exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError("REPO contains a non-Git runtime. Select a new writable checkout directory.")
    REPO.parent.mkdir(parents=True,exist_ok=True)
    subprocess.run(["git","clone","--quiet","--no-checkout",REPO_URL,str(REPO)],check=True)
    cloned_now = True
dirty = subprocess.check_output(["git","-C",str(REPO),"status","--porcelain","--untracked-files=no"],text=True)
if dirty.strip() and not cloned_now:
    raise RuntimeError("Tracked source changes in REPO; select a fresh directory instead of overwriting them.")
available = subprocess.run(["git","-C",str(REPO),"rev-parse","--verify",REVISION+"^{commit}"],capture_output=True)
if available.returncode:
    subprocess.run(["git","-C",str(REPO),"fetch","--quiet","origin",REVISION],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--quiet","--detach",REVISION],check=True)
actual = subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
if len(REVISION) != 40 or actual != REVISION:
    raise RuntimeError("Notebook must select the exact published40-character source commit.")
if not complete(REPO):
    raise RuntimeError("Selected source commit does not contain the required implementation.")
print("Source:",REPO_URL,"commit:",actual)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO/"requirements-tgsc.txt")], check=True)
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime; notebook training does not silently fall back to CPU."
print("torch:", torch.__version__, "GPU:", torch.cuda.get_device_name(GPU))
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Reuse only the teacher from an attached old Dataset; do not read its crop bank.
if TEACHER_CHECKPOINT is None:
    matches = [p for p in Path("/kaggle/input").rglob("ckpt_epoch_240.pth") if p.parent.name == TEACHER+"_vanilla"]
    if len(matches) > 1:
        raise RuntimeError("Multiple teacher copies found; set TEACHER_CHECKPOINT explicitly.")
    if matches:
        TEACHER_CHECKPOINT = matches[0]
    else:
        subprocess.run([sys.executable, str(REPO/"tools/download_epkd_teacher.py"), "--teacher", TEACHER,
                        "--output-dir", str(OUTPUT/"teacher_weights")], check=True)
        TEACHER_CHECKPOINT = OUTPUT/"teacher_weights/cifar_teachers"/(TEACHER+"_vanilla")/"ckpt_epoch_240.pth"
assert Path(TEACHER_CHECKPOINT).is_file()
# Set DATA_ROOT to a folder containing cifar-100-python/, including an attached read-only Dataset.
# Otherwise torchvision downloads the original archive once, shared by all methods/seeds here.
from torchvision.datasets import CIFAR100
if not (DATA_ROOT/"cifar-100-python/train").exists():
    CIFAR100(str(DATA_ROOT), train=True, download=True)
print("Teacher:", TEACHER_CHECKPOINT, "Original CIFAR:", DATA_ROOT)


In [ ]:
# Explicit full-run launcher: no synthetic data, no stored crop banks.
def run(command, config, seed, output, extra=()):
    cmd = [sys.executable, str(REPO/"tools/run_epkd.py"), command,
           "--config", str(config), "--teacher-checkpoint", str(TEACHER_CHECKPOINT),
           "--data-root", str(DATA_ROOT), "--output", str(output), "--gpu", str(GPU),
           "--seed", str(seed), "--eval-every", str(EVAL_EVERY), "--download", *extra]
    if TEST_DATA:
        cmd.extend(["--test-data", str(TEST_DATA)])
    subprocess.run(cmd, cwd=REPO, check=True)

for name in CONFIGS:
    config = REPO/"configs/epkd/cifar100"/name
    for seed in SEEDS:
        output = OUTPUT/(config.stem+f"_seed{seed}")
        latest = output/"latest.pt"
        run("check", config, seed, output, ["--limit", "16"])
        run("train", config, seed, output, ["--resume", str(latest)] if latest.exists() else [])
        run("evaluate", config, seed, output, ["--export", str(output/"inference_best.pt")])
        from IPython.display import display, Image
        display(Image(filename=str(output/"training_curves.png")))
